# Session 6: Function Calling & Tool Use

## Objectives
- Understand how LLMs can call external functions
- Define tool schemas for the OpenAI API
- Handle function call responses and return results
- Build a multi-tool assistant

**Duration:** 40 minutes | **Level:** Medium

**Why this matters:** Function calling lets LLMs interact with the real world â€” databases, APIs, calculators, and any code you write.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"Setup complete! Model: {MODEL}")

## 1. How Function Calling Works

The flow is:
1. **You** define available functions (tools) with their schemas
2. **LLM** decides if/which function to call and generates the arguments
3. **You** execute the function with those arguments
4. **You** send the result back to the LLM
5. **LLM** generates a natural language response

```
User Message â†’ LLM â†’ "Call get_weather(city='Paris')" â†’ Your Code â†’ Result â†’ LLM â†’ Final Answer
```

**Important:** The LLM does NOT execute functions â€” it only decides which to call and with what arguments. YOU execute them.

## 2. Defining Tools

Each tool is defined with:
- `name`: The function name
- `description`: What the function does (helps the LLM decide when to use it)
- `parameters`: JSON Schema describing the expected arguments

In [ ]:
# Step 1: Define your actual Python functions
def get_weather(city):
    """Simulate getting weather data (in real app, call a weather API)."""
    weather_data = {
        "Paris": {"temp": 18, "condition": "Partly Cloudy", "humidity": 65},
        "Tokyo": {"temp": 24, "condition": "Sunny", "humidity": 45},
        "New York": {"temp": 12, "condition": "Rainy", "humidity": 80},
        "London": {"temp": 14, "condition": "Overcast", "humidity": 75},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown", "humidity": 50})
    return json.dumps({"city": city, **data})

def calculate(expression):
    """Safely evaluate a math expression."""
    # Only allow safe math operations
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return json.dumps({"error": "Invalid expression"})
    try:
        result = eval(expression)  # Safe because we validated the input
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})

# Step 2: Define the tool schemas for the API
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g., 'Paris' or 'Tokyo'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calculate a mathematical expression. Supports +, -, *, /, parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, e.g., '(15 + 23) * 2'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print(f"Defined {len(tools)} tools: {[t['function']['name'] for t in tools]}")

## 3. Making a Function Call

When the LLM decides to use a tool, the response contains a `tool_calls` field instead of regular content.

In [ ]:
# Send a message that requires a tool
messages = [
    {"role": "system", "content": "You are a helpful assistant with access to weather and calculator tools."},
    {"role": "user", "content": "What's the weather like in Tokyo?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools
)

# Check if the model wants to call a function
message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Content: {message.content}")
print(f"Tool calls: {message.tool_calls}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"\nFunction to call: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")

## 4. The Complete Function Calling Flow

After the LLM requests a function call, we must:
1. Execute the function
2. Send the result back as a `tool` message
3. Let the LLM generate the final response

In [ ]:
# Map function names to actual Python functions
available_functions = {
    "get_weather": get_weather,
    "calculate": calculate
}

def run_with_tools(user_message):
    """Complete function calling flow."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user", "content": user_message}
    ]
    
    # First API call â€” the LLM may request tool calls
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    
    message = response.choices[0].message
    
    # If no tool calls, return the direct response
    if not message.tool_calls:
        return message.content
    
    # Process each tool call
    messages.append(message)  # Add assistant's tool call message
    
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        print(f"  Calling: {function_name}({function_args})")
        
        # Execute the function
        function = available_functions[function_name]
        result = function(**function_args)
        
        print(f"  Result: {result}")
        
        # Add the tool result to messages
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
    
    # Second API call â€” LLM generates final response using tool results
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    
    return final_response.choices[0].message.content

# Test it!
print("\n" + run_with_tools("What's the weather in Paris?"))

In [ ]:
# Test with calculator
print(run_with_tools("What is (145 + 287) * 3?"))

In [ ]:
# Test with a question that doesn't need tools
print(run_with_tools("What is the capital of Germany?"))

## 5. Handling Multiple Tool Calls

The LLM can request multiple function calls in a single response (parallel tool calls).

In [ ]:
# This should trigger multiple tool calls
result = run_with_tools("Compare the weather in Tokyo and London. Also, what is 72 * 1.8 + 32?")
print("\n" + result)

## 6. Adding a New Tool

Let's add a third tool to show how easy it is to extend.

In [ ]:
# New function: Get current date/time
from datetime import datetime

def get_current_datetime(timezone="UTC"):
    """Get the current date and time."""
    now = datetime.now()
    return json.dumps({
        "datetime": now.strftime("%Y-%m-%d %H:%M:%S"),
        "timezone": timezone
    })

# Add to tools list
tools.append({
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Get the current date and time.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "The timezone, e.g., 'UTC', 'EST'"
                }
            },
            "required": []
        }
    }
})

# Update available functions
available_functions["get_current_datetime"] = get_current_datetime

# Test the new tool
result = run_with_tools("What time is it right now?")
print("\n" + result)

## Exercise: Build Your Own Tool

Add a `search_products` tool that searches a simple product database.

In [ ]:
# Product database
products = [
    {"name": "Laptop Pro 15", "category": "electronics", "price": 1299, "rating": 4.5},
    {"name": "Wireless Mouse", "category": "electronics", "price": 29, "rating": 4.2},
    {"name": "Standing Desk", "category": "furniture", "price": 499, "rating": 4.7},
    {"name": "Noise-Cancelling Headphones", "category": "electronics", "price": 349, "rating": 4.8},
    {"name": "Ergonomic Chair", "category": "furniture", "price": 699, "rating": 4.6},
]

def search_products(category=None, max_price=None):
    """Search products by category and/or max price."""
    results = products
    if category:
        results = [p for p in results if p["category"] == category]
    if max_price:
        results = [p for p in results if p["price"] <= max_price]
    return json.dumps(results)

# Add the tool schema
tools.append({
    "type": "function",
    "function": {
        "name": "search_products",
        "description": "Search the product catalog by category and/or maximum price.",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Product category: 'electronics' or 'furniture'",
                    "enum": ["electronics", "furniture"]
                },
                "max_price": {
                    "type": "number",
                    "description": "Maximum price in dollars"
                }
            },
            "required": []
        }
    }
})

available_functions["search_products"] = search_products

# Test it
result = run_with_tools("Show me electronics under $100")
print("\n" + result)

print("\n" + "="*50 + "\n")

result = run_with_tools("What's the best-rated product you have?")
print("\n" + result)

## Summary

**Function calling flow:**
1. Define tools with JSON schemas
2. Send tools to the API with your messages
3. LLM decides when to use tools and generates arguments
4. You execute the functions and return results
5. LLM generates the final response

**Key takeaways:**
- The LLM only *decides* which function to call â€” you execute it
- Good tool descriptions help the LLM choose correctly
- Always validate function arguments before execution
- The LLM can call multiple tools in parallel

**Next session:** Building autonomous agents that use tools in a loop!